# Introduction (TBD)

# Libraries and Global Variables

In [ ]:
import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport
from pathlib import Path

In [ ]:
#Seed for ML Algorithms
SEED = 81

#Paths
PROJECT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists()
)
GENERATED_REPORTS = (PROJECT / "reports" / "generated")

DIABETES_CSV = PROJECT / "data" / "raw" / "diabetic_data.csv"
IDS_MAPING_CSV = PROJECT / "data" / "raw" / "IDS_mapping.csv"


## Diabetes fact table

In [ ]:
#Read diabetes CSV
df_diabetes = pd.read_csv(DIABETES_CSV, delimiter=';')
df_diabetes.dtypes.to_csv(PROJECT / "data" / "raw" / "diabetes_dtypes.csv")

#CATEGORIZE VARIABLES

## NUMERIC
NUMERIC = [
    "encounter_id",
    "patient_nbr",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

CATEGORICAL = [
    "race",
    "gender",
    "age",
    "weight",
    "payer_code",
    "medical_specialty",
    "diag_1",
    "diag_2",
    "diag_3",
    "max_glu_serum",
    "A1Cresult",
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "acetohexamide",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "troglitazone",
    "tolazamide",
    "examide",
    "citoglipton",
    "insulin",
    "glyburide-metformin",
    "glipizide-metformin",
    "glimepiride-pioglitazone",
    "metformin-rosiglitazone",
    "metformin-pioglitazone",
    "change",
    "diabetesMed",
    "readmitted"
]

## IDS Mapping dim tables
TODO: Add small explanation behind logic of generator 

In [ ]:
def process_ids_tables():
    #Read IDS mapping CSV
    df_ids_map = pd.read_csv(IDS_MAPING_CSV, header=None, names=range(2), delimiter=';')

    #Define table names
    table_names = ['admission_type', 'discharge_disposition_id', 'admission_source_id']

    groups = df_ids_map[0].isin(table_names).cumsum()

    tables = {g.iloc[0,0]: g.iloc[0:-1] for k,g in df_ids_map.groupby(groups)}

    for name, table in tables.items():
        #Create dataframe
        df = pd.DataFrame(table)

        #Promote first line to header
        df.columns = df.iloc[0]
        df = df.iloc[1:].reset_index(drop=True)

        yield df

ids_mapping_dataframes = process_ids_tables()

#Use generator to create our dataframes for each table

#Admission Types
df_admission_type = next(ids_mapping_dataframes)

#Discharge Disposition
df_discharge_disposition = next(ids_mapping_dataframes)

#Admission Source
df_admission_source = next(ids_mapping_dataframes)

# EDA

Just for starters, we'll create a Profile Report for the diabetes main fact table

In [ ]:
profile = ProfileReport(
    df_diabetes, title="Diabetes Data Profile Report"
)
profile.config.html.inline = True

file_name = "profile-report-diabests.html"
path_to_html = Path(PROJECT / "reports" / "generated" / file_name)

if not path_to_html.exists():
    profile.to_file(GENERATED_REPORTS / file_name)